# Sentiment Analysis for Customer Feedback – Product and Service Improvements with Precision

## Data Cleaning & Preprocessing
- Data Cleaning and Preprocessing: Perform tokenisation, lemmatisation, text vectorisation using TF-IDF and contextual embeddings, and balance the dataset.

### Data Cleaning

- Remove duplicates: drop exact repeated entries, and check for near-duplicates (same text with minor formatting differences, or paraphrased versions using similarity scoring).
Handle missing data: drop rows with missing text (since it usually can't be guessed), drop or impute missing labels, impute missing metadata fields, or flag missingness if it's meaningful.
Normalize text: lowercase everything, strip out URLs, HTML tags, punctuation, and extra whitespace, fix encoding/unicode issues, and expand contractions ("don't" → "do not").

### Data Preprocessing

- Tokenisation : split text into words or subword units. Word-level tokenisation works for traditional models; subword tokenisation (used by transformer models like BERT) handles unfamiliar words better.
Stop-word removal: strip common filler words ("the", "is", "and") for traditional models like TF-IDF. This step is usually skipped for transformer/embedding models, since they rely on full sentence structure.
- Lemmatisation: reduce words to their dictionary base form (e.g. "running," "ran" → "run"), which shrinks vocabulary size and improves generalisation.
- Vectorisation: convert text into numbers so models can use it:

- TF-IDF: weighs words by importance across the dataset; simple, fast, and interpretable, but has no sense of meaning or context (can't tell "bank" the river from "bank" the financial institution).
Contextual embeddings: dense vectors from pretrained transformer models that capture actual meaning and context; more powerful but computationally heavier.

# Import the Required

In [29]:
"""
Import statements for a machine learning project focused on text classification.
Organized by functionality for better maintainability.
"""

from typing import Any
import warnings

# Core data science libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Text processing libraries
import re
import string
import nltk
from nltk.corpus import stopwords
# import ftfy  # Commented out - install with: pip install ftfy or conda install ftfy
# Add the missing import at the top of your code
from langdetect import detect, LangDetectException

# Classical ML libraries
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

# Deep learning libraries
import torch
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

# Utility libraries
from collections import defaultdict, Counter
import joblib
import streamlit as st
import ftfy
# Import the spaCy library for natural language processing
import spacy
# Configure warnings
warnings.filterwarnings('ignore')  # Suppress compatibility warnings


# Import Dataset From working Directory

In [3]:
# EXPLAINABILITY NOTE:
# This cell loads the raw customer review CSV into a DataFrame.
# All later cleaning, exploratory analysis, and modelling steps start from this data.

# Load review data from CSV file into a pandas DataFrame
df = pd.read_csv("review_data.csv")
# Display the first 5 rows of the DataFrame to inspect the data structure
df.head()

,review_id,product_category,timestamp,country,rating,review,sentiment
0,REV-50BCBCD9,Sports,2024-09-16 13:44:26+00:00,US,1,"I registered on the website, tried to order a ...",Negative
1,REV-6D2B2651,Toys,2024-09-16 18:26:46+00:00,GB,1,Had multiple orders one turned up and driver h...,Negative
2,REV-F7E80372,Toys,2024-09-16 21:47:39+00:00,GB,1,I informed these reprobates that I WOULD NOT B...,Negative
3,REV-ED2B173F,Sports,2024-09-17 07:15:49+00:00,AU,1,I have bought from Amazon before and no proble...,Negative
4,REV-E48A7AB9,Fashion,2024-09-16 18:37:17+00:00,GB,1,If I could give a lower rate I would! I cancel...,Negative


# Data Quality Checks and Audit

In [4]:
# EXPLAINABILITY NOTE:
# This function checks basic data quality issues before modelling.
# It reports empty reviews and category consistency so data problems are visible early.

def data_quality_check(df):
    # Initialize quality report list
    quality_report = []
    total_rows = len(df)
    
    # Assuming empty_reviews was calculated earlier in the function
    # You'll need to add the actual calculation for empty_reviews
    empty_reviews = df["review"].isnull().sum() + (df["review"] == "").sum()
    
    # Add empty reviews check to quality report
    quality_report.append({
        "Check": "Empty Reviews",  # Fixed: Added proper key
        "Column": "review",
        "Issue_Count": empty_reviews,
        "Percentage": round((empty_reviews/total_rows)*100, 2),
        "Status": "Fail" if empty_reviews > 0 else "Pass"
    })

    # ======================================================================
    # 8. CATEGORY CONSISTENCY
    # ======================================================================
    # Check if product categories have inconsistent casing by comparing
    # unique count of lowercase categories vs original categories
    inconsistent_categories = (
        df["product_category"].str.lower().nunique()
        != df["product_category"].nunique()
    )

    # Add category consistency check results to quality report
    quality_report.append({
        "Check": "Category Consistency",
        "Column": "product_category",
        "Issue_Count": int(inconsistent_categories),
        "Percentage": None,
        "Status": "Fail" if inconsistent_categories else "Pass"
    })

    # ======================================================================
    # CREATE FINAL REPORT
    # ======================================================================
    # Convert quality report list to DataFrame for better presentation
    report = pd.DataFrame(quality_report)

    # Calculate total number of issues across all checks
    total_issues = report["Issue_Count"].sum()

    # Calculate overall data quality score as percentage
    # Formula: (1 - total_issues / total_possible_issues) * 100
    quality_score = round(
        (1 - total_issues/(total_rows*len(df.columns))) * 100,
        2
    )

    # Display the complete data quality report
    print("\nDATA QUALITY REPORT")
    print(report)

    # Display overall quality score with formatting
    print("\n"+"="*70)
    print("OVERALL DATA QUALITY SCORE:", quality_score, "%")
    print("="*70)

    # Return the report DataFrame for further use
    return report

In [5]:
# EXPLAINABILITY NOTE:
# This cell runs the data quality function and stores the report.
# The report helps decide whether cleaning is needed before modelling.

# Generate an automated data quality report for the DataFrame
quality_report = data_quality_check(df)


DATA QUALITY REPORT
                  Check            Column  Issue_Count  Percentage Status
0         Empty Reviews            review            0         0.0   Pass
1  Category Consistency  product_category            0         NaN   Pass

OVERALL DATA QUALITY SCORE: 100.0 %


# Data Cleaning Pipeline

In [6]:
# EXPLAINABILITY NOTE:
# This function applies safe data cleaning steps to dates and review text.
# It also records a cleaning log so each transformation is auditable.

def clean_data(df):
    # Initialize cleaning log to track all cleaning steps
    cleaning_log = []
    
    # Count initial number of rows before cleaning
    initial_rows = len(df)
    
    # Count invalid dates before conversion
    invalid_dates_before = df["timestamp"].isnull().sum()
    
    # Convert timestamp column to datetime format
    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        errors="coerce"
    )

    # Count invalid dates after conversion (newly created NaT values)
    invalid_dates_after = df["timestamp"].isnull().sum()

    # Log the timestamp conversion step
    # Track how many additional invalid dates were created during conversion
    cleaning_log.append({
        "Cleaning_Step": "Convert Timestamp Format",
        "Rows_Affected": invalid_dates_after - invalid_dates_before
    })

    # ==========================================================
    # 6. FIX TEXT ENCODING ISSUES
    # ==========================================================
    # Fix text encoding issues in review column
    # Convert to string and apply ftfy library to fix mojibake and encoding problems
    df["review"] = (
        df["review"]
        .astype(str)
        .apply(ftfy.fix_text)
    )

    # Log the text encoding fix step
    # All rows are potentially affected by encoding fixes
    cleaning_log.append({
        "Cleaning_Step": "Fix Text Encoding Issues",
        "Rows_Affected": len(df)
    })

    # ==========================================================
    # CREATE CLEANING REPORT
    # ==========================================================
    # Count final number of rows after all cleaning steps
    final_rows = len(df)

    # Calculate total rows removed during cleaning process
    rows_removed = initial_rows - final_rows

    # Convert cleaning log to DataFrame for better presentation
    cleaning_report = pd.DataFrame(cleaning_log)

    # Display comprehensive cleaning report
    print("\nCLEANING REPORT")
    print(cleaning_report)

    # Display summary statistics of the cleaning process
    print("\n"+"="*70)
    print("Rows Before Cleaning:", initial_rows)
    print("Rows After Cleaning:", final_rows)
    print("Rows Removed:", rows_removed)
    print("="*70)

    # Return cleaned DataFrame and detailed cleaning report
    return df, cleaning_report

In [16]:
# EXPLAINABILITY NOTE:
# This cell applies the cleaning function and keeps both the cleaned data
# and the cleaning report for later inspection.

clean_df, cleaning_report = clean_data(df)


CLEANING REPORT
              Cleaning_Step  Rows_Affected
0  Convert Timestamp Format              0
1  Fix Text Encoding Issues          21055

Rows Before Cleaning: 21055
Rows After Cleaning: 21055
Rows Removed: 0


# Multilingual Language Detection Pipeline

In [21]:
# EXPLAINABILITY NOTE:
# This cell detects review language and creates language summary outputs.
# The pipeline includes error handling so short or unclear reviews do not break execution.

# ======================================================================
# 1. TEXT CLEANING FUNCTION
# Purpose: Remove noise that reduces language detection accuracy
# ======================================================================
def clean_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower()

    # Remove URLs (not useful for language detection)
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove punctuation / emojis while keeping accented characters
    text = re.sub(r"[^a-zA-ZÀ-ÿ0-9\s]", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ======================================================================
# 2. LANGUAGE DETECTION FUNCTION
# Purpose: Safely detect language with error handling
# ======================================================================
def detect_language(text):
    try:
        if not text or len(text.strip()) < 3:
            return "unknown"
        return detect(text)
    except LangDetectException:
        return "unknown"
    except Exception:
        return "error"


# ======================================================================
# 3. AUTO TEXT COLUMN DETECTION
# Purpose: Make pipeline dataset-agnostic (no hardcoding needed)
# ======================================================================
def auto_detect_text_column(df):
    possible_columns = [
        "review_text", "review", "text",
        "content", "comment", "message"
    ]

    for col in possible_columns:
        if col in df.columns:
            return col

    raise ValueError(f"No valid text column found. Available columns: {list(df.columns)}")


# ======================================================================
# 4. VALIDATION SAMPLER
# Purpose: Manual QA of language detection quality
# ======================================================================
def get_validation_samples(df, lang_col="detected_language", text_col="text", n=3):
    return (
        df.groupby(lang_col)
        .apply(lambda x: x.sample(min(n, len(x)), random_state=42))
        .reset_index(drop=True)[[text_col, lang_col]]
    )


# ======================================================================
# 5. FULL PIPELINE (FIXED + PRODUCTION SAFE)
# ======================================================================
def run_language_pipeline(df, text_column=None, output_column="detected_language", clean=True):

    df = df.copy()

    # Step 1: Auto-detect text column if not provided
    if text_column is None:
        text_column = auto_detect_text_column(df)

    # Step 2: Clean text (optional but improves accuracy)
    if clean:
        df["_clean_text"] = df[text_column].apply(clean_text)
        source_col = "_clean_text"
    else:
        source_col = text_column

    # Step 3: Language detection
    df[output_column] = df[source_col].apply(detect_language)

    # Step 4: Data quality flag
    df["is_valid_detection"] = ~df[output_column].isin(["unknown", "error"])

    # Step 5: FIXED language distribution (robust version)
    counts = df[output_column].value_counts(normalize=True)

    language_summary = counts.reset_index()
    language_summary.columns = ["language", "proportion"]

    language_summary["percentage"] = language_summary["proportion"] * 100

    return df, language_summary, text_column


# ======================================================================
# 6. EXECUTION BLOCK
# ======================================================================
df, language_summary, used_column = run_language_pipeline(df)

print("======================================================")
print("TEXT COLUMN USED:", used_column)
print("======================================================")

print("\nLANGUAGE DISTRIBUTION")
print(language_summary)

print("\nVALIDATION SAMPLES")
print(get_validation_samples(df, text_col=used_column))

# Optional: filter for modelling (e.g., English-only sentiment model)
df_filtered = df[df["detected_language"] == "en"]

TEXT COLUMN USED: review

LANGUAGE DISTRIBUTION
   language  proportion  percentage
0        en    0.990501   99.050107
1        it    0.001995    0.199478
2        fr    0.000997    0.099739
3        da    0.000807    0.080741
4        af    0.000712    0.071242
5        nl    0.000665    0.066493
6        ca    0.000617    0.061743
7        hr    0.000570    0.056994
8        no    0.000522    0.052244
9        ro    0.000332    0.033246
10       es    0.000285    0.028497
11       cy    0.000285    0.028497
12       so    0.000285    0.028497
13       hu    0.000237    0.023747
14       sl    0.000237    0.023747
15       sq    0.000142    0.014248
16       id    0.000142    0.014248
17       et    0.000142    0.014248
18       sk    0.000095    0.009499
19       sv    0.000095    0.009499
20       sw    0.000095    0.009499
21       pl    0.000095    0.009499
22       de    0.000047    0.004749
23       fi    0.000047    0.004749
24       cs    0.000047    0.004749

VALIDATION SAMP

# Text Preprocessing Pipeline

In [26]:
# EXPLAINABILITY NOTE:
# This cell prepares model-ready text by cleaning, tokenising, removing stopwords,
# and lemmatising where a spaCy language model is available.

# Ensure stopwords are available
nltk.download("stopwords", quiet=True)

# Load spaCy model if installed; fall back to a lightweight blank English pipeline.
try:
    nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
except OSError:
    nlp = spacy.blank("en")

stop_words = set(stopwords.words("english"))


def clean_text(text):
    """Clean review text while preserving useful word content."""
    if pd.isna(text):
        return ""

    text = ftfy.fix_text(str(text)).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def preprocess_text(text):
    """Tokenise, remove stopwords, and use lemmas when available."""
    doc = nlp(text)
    tokens = []

    for token in doc:
        token_text = token.text.strip().lower()
        lemma = token.lemma_.strip().lower() if token.lemma_ else token_text

        if token_text in stop_words:
            continue
        if token.is_punct or token.is_space:
            continue
        if len(token_text) < 2:
            continue

        tokens.append(lemma)

    return " ".join(tokens)


def auto_detect_text_column(df):
    possible_columns = ["review_text", "review", "text", "content", "comment", "message"]

    for col in possible_columns:
        if col in df.columns:
            return col

    raise ValueError(f"No valid text column found. Available columns: {list(df.columns)}")


def run_nlp_pipeline(df, text_column=None):
    """End-to-end NLP preprocessing pipeline."""
    df = df.copy()

    if text_column is None:
        text_column = auto_detect_text_column(df)

    df["_clean_text"] = df[text_column].apply(clean_text)
    df["processed_review"] = df["_clean_text"].apply(preprocess_text)

    return df, text_column


df, used_column = run_nlp_pipeline(df)

print("======================================================")
print("TEXT COLUMN USED:", used_column)
print("======================================================")
print(df[[used_column, "processed_review"]].head())

TEXT COLUMN USED: review
                                              review  \
0  I registered on the website, tried to order a ...   
1  Had multiple orders one turned up and driver h...   
2  I informed these reprobates that I WOULD NOT B...   
3  I have bought from Amazon before and no proble...   
4  If I could give a lower rate I would! I cancel...   

                                    processed_review  
0  registered website tried order laptop entered ...  
1  multiple orders one turned driver phone door n...  
2  informed reprobates would going visit sick rel...  
3  bought amazon problems happy service price ama...  
4  could give lower rate would cancelled amazon p...  


# Sentiment Driver Discovery
### Identify key themes influencing sentiment

In [27]:
# EXPLAINABILITY NOTE:
# This cell extracts frequent two-word phrases from positive and negative reviews.
# Bigrams help explain which review phrases are driving sentiment labels.

def auto_detect_text_column_safe(dataframe):
    """Find the most likely review text column for analysis/modelling cells."""
    possible_columns = ["processed_review", "review_text", "review", "text", "content", "comment", "message"]

    for col in possible_columns:
        if col in dataframe.columns:
            return col

    raise ValueError(f"No valid text column found. Available columns: {list(dataframe.columns)}")


# ======================================================================
# BIGRAM ANALYSIS USING THE REAL REVIEW DATA
# ======================================================================

text_col = "processed_review" if "processed_review" in df.columns else auto_detect_text_column_safe(df)
analysis_df = df[[text_col, "sentiment"]].copy()
analysis_df[text_col] = analysis_df[text_col].astype(str).str.strip()
analysis_df["sentiment"] = analysis_df["sentiment"].astype(str).str.lower().str.strip()
analysis_df = analysis_df[analysis_df[text_col] != ""]

positive_reviews = analysis_df.loc[analysis_df["sentiment"] == "positive", text_col].tolist()
negative_reviews = analysis_df.loc[analysis_df["sentiment"] == "negative", text_col].tolist()


def get_top_bigrams(reviews, max_features=15, min_df=2):
    valid_reviews = [review for review in reviews if len(str(review).strip()) > 0]

    if not valid_reviews:
        return pd.DataFrame(columns=["bigram", "count"])

    vectorizer = CountVectorizer(
        ngram_range=(2, 2),
        max_features=max_features,
        min_df=min_df,
        stop_words="english",
    )

    matrix = vectorizer.fit_transform(valid_reviews)
    return (
        pd.DataFrame({
            "bigram": vectorizer.get_feature_names_out(),
            "count": matrix.sum(axis=0).A1,
        })
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )


positive_bigrams = get_top_bigrams(positive_reviews)
negative_bigrams = get_top_bigrams(negative_reviews)

print("\nTop Positive Bigrams")
print(positive_bigrams)

print("\nTop Negative Bigrams")
print(negative_bigrams)


Top Positive Bigrams
              bigram  count
0   customer service    938
1        review text    388
2       amazon prime    346
3        love amazon    285
4         amazon com    197
5         use amazon    150
6    amazon customer    143
7      fast delivery    141
8      great service    133
9       prime member    125
10      amazon great    124
11      amazon years    115
12   online shopping    114
13      using amazon    108
14     free shipping    103

Top Negative Bigrams
               bigram  count
0    customer service   5992
1     amazon customer   1107
2        amazon prime   1105
3           gift card    803
4         credit card    736
5        day delivery    628
6    prime membership    600
7        prime member    559
8      amazon account    481
9    contacted amazon    423
10         use amazon    402
11       order amazon    375
12    amazon delivery    364
13  customer services    361
14      delivery date    361


### Save Preprocessed Data

In [28]:
# ======================================================================
# SAVE PREPROCESSED DATA
# Purpose: Store cleaned and processed data for future analysis
# ======================================================================

# Export the processed DataFrame to CSV format without row indices
# This creates a backup of the cleaned data for reproducibility and future use
df.to_csv("processed_sentiment_data.csv", index=False)